# Analysis of Mixed-Precision Training and Inference

This notebook analyzes the results of mixed-precision deep learning experiments. The goal is to understand the trade-offs between performance (throughput, memory) and accuracy (F1-score) across different hardware (GPUs), models, and numerical precisions (FP32, BF16, FP16).

## 1. Setup and Data Loading

First, we import the necessary libraries and define the paths to our experiment data. We'll load all the JSON files containing the metrics from our runs.

In [3]:
!pip3 install plotly

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/9.9 MB ? eta -:--:--Downloading plotly-6.4.0-py3-none-any.whl (9.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 19.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 19.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import os
import json
import re
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Define the root directory for the experiment data
DATA_DIR = "train_experiment_data/"


In [5]:
def find_json_files(directory):
    """Find all JSON files in a directory and its subdirectories."""
    json_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".json"):
                json_files.append(os.path.join(root, file))
    return json_files

def load_json_data(file_path):
    """Load a single JSON file and return its content."""
    with open(file_path, 'r') as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            print(f"Warning: Could not decode JSON from {file_path}")
            return None

# Load all JSON files
all_files = find_json_files(DATA_DIR)
all_data = []
for f in all_files:
    data = load_json_data(f)
    if data:
        # Add filename for metadata extraction
        data['filename'] = os.path.basename(f)
        all_data.append(data)

print(f"Loaded {len(all_data)} JSON files.")

Loaded 486 JSON files.


## 2. Data Preprocessing and Feature Extraction

Next, we'll process the loaded data. We'll parse the filenames and the JSON content to extract key information such as the GPU, model, precision, and batch size. This will be compiled into a clean pandas DataFrame.

In [6]:
def parse_filename(filename):
    """Parse the filename to extract experiment parameters."""
    parts = filename.split('_')
    
    # Simple parsing logic, may need adjustment based on filename conventions
    # Example: nvidia_infer_bert-large_bf16_bs128_r1.json
    # Example: amd_train_resnet50_fp16_bs64_e10_r1.json
    
    info = {
        'gpu_type': 'unknown',
        'model': 'unknown',
        'precision': 'unknown',
        'batch_size': 0,
        'experiment_type': 'unknown'
    }
    
    if 'nvidia' in filename or '4090' in filename or 'L40S' in filename:
        info['gpu_type'] = 'NVIDIA'
    elif 'amd' in filename or '7900' in filename:
        info['gpu_type'] = 'AMD'
    elif 'grace' in filename:
        info['gpu_type'] = 'Grace'

    if 'infer' in filename:
        info['experiment_type'] = 'inference'
    elif 'train' in filename:
        info['experiment_type'] = 'training'
        
    model_match = re.search(r'(bert-large|gpt2|resnet50)', filename)
    if model_match:
        info['model'] = model_match.group(1)
        
    precision_match = re.search(r'(bf16|fp16|fp32)', filename)
    if precision_match:
        info['precision'] = precision_match.group(1)
        
    bs_match = re.search(r'bs(\d+)', filename)
    if bs_match:
        info['batch_size'] = int(bs_match.group(1))
        
    return info

# Create DataFrame
df = pd.DataFrame(all_data)

# Expand filename into columns
df_meta = df['filename'].apply(parse_filename).apply(pd.Series)
df = pd.concat([df, df_meta], axis=1)

# Extract metrics from the 'results' dictionary if it exists
if 'results' in df.columns:
    df_results = df[df['results'].notna()]['results'].apply(pd.Series)
    df = pd.concat([df.drop('results', axis=1), df_results], axis=1)

# Rename columns for clarity and consistency
df = df.rename(columns={
    'samples_per_second': 'throughput',
    'eval_f1-score': 'f1_score' # Adjust if the key is different
})

# Display the first few rows of the processed data
df.head()

,model,precision,pretrained,device,batch_size,num_batches,warmup_batches,total_samples,total_inference_time_sec,throughput_samples_per_sec,...,p95_latency_ms,p99_latency_ms,peak_memory_mb,success,filename,gpu_type,model,precision,batch_size,experiment_type
0,bert-large,fp16,True,cuda,1,50.0,5.0,50.0,0.842574,59.341944,...,18.638694,23.095193,1867.809082,1,nvidia_infer_bert-large_fp16_bs1_r1.json,NVIDIA,bert-large,fp16,1,inference
1,gpt2,bf16,True,cuda,8,50.0,5.0,400.0,0.582562,686.622232,...,12.103462,22.324779,1019.142578,1,nvidia_infer_gpt2_bf16_bs8_r1.json,NVIDIA,gpt2,bf16,8,inference
2,bert-large,fp16,True,cuda,32,50.0,5.0,1600.0,1.234719,1295.841612,...,24.810183,28.307621,1983.879883,1,nvidia_infer_bert-large_fp16_bs32_r1.json,NVIDIA,bert-large,fp16,32,inference
3,resnet50,bf16,True,cuda,48,50.0,5.0,2400.0,0.546820,4389.009610,...,13.156080,17.115548,374.746094,1,nvidia_infer_resnet50_bf16_bs48_r1.json,NVIDIA,resnet50,bf16,48,inference
4,gpt2,fp32,True,cuda,1,50.0,5.0,50.0,0.468672,106.684356,...,9.840667,12.341492,563.674805,1,nvidia_infer_gpt2_fp32_bs1_r1.json,NVIDIA,gpt2,fp32,1,inference


## 3. Throughput vs. Batch Size Analysis

Here, we visualize how throughput scales with the batch size for different GPUs, models, and precisions. This helps in identifying the optimal batch size for maximizing performance.

In [8]:
fig = px.line(
    df.sort_values('batch_size'),
    x='batch_size',
    y='throughput',
    color='precision',
    facet_row='model',
    facet_col='gpu_type',
    title='Throughput vs. Batch Size',
    markers=True
)
fig.update_layout(height=800)
fig.show()

ValueError: The column label 'batch_size' is not unique.

## 4. Accuracy/F1-Score Analysis vs. Precision

This section compares the final F1-score across different precision settings. This is crucial to ensure that performance gains from mixed precision do not come at the cost of a significant drop in model accuracy. We will look at the results for a large, stable batch size.

In [9]:
# We need to select a representative batch size to compare F1 scores
# Let's find a batch size that is common across many runs, or just pick a large one
stable_batch_size = df[df['batch_size'] > 0]['batch_size'].mode()[0]

df_accuracy = df[df['batch_size'] == stable_batch_size]

if 'f1_score' in df_accuracy.columns:
    fig = px.bar(
        df_accuracy,
        x='precision',
        y='f1_score',
        color='gpu_type',
        barmode='group',
        facet_col='model',
        title=f'F1-Score vs. Precision (Batch Size = {stable_batch_size})'
    )
    fig.show()
else:
    print("F1 score data not found in the processed dataframe.")

ValueError: cannot reindex on an axis with duplicate labels

## 5. Hardware Comparison: Throughput

Let's compare the maximum throughput achieved on different GPUs for each model and precision. This provides a direct comparison of hardware performance.

In [10]:
# Find the max throughput for each group
max_throughput_df = df.loc[df.groupby(['gpu_type', 'model', 'precision'])['throughput'].idxmax()]

fig = px.bar(
    max_throughput_df,
    x='gpu_type',
    y='throughput',
    color='precision',
    barmode='group',
    facet_col='model',
    title='Max Throughput Comparison Across GPUs'
)
fig.show()

ValueError: Grouper for 'model' not 1-dimensional

## 6. Hardware Comparison: Accuracy

We need to ensure that the model's accuracy is consistent across different hardware platforms when using the same precision format. This validates the consistency of the experimental setup.

In [ ]:
if 'f1_score' in df_accuracy.columns:
    fig = px.bar(
        df_accuracy,
        x='gpu_type',
        y='f1_score',
        color='precision',
        barmode='group',
        facet_col='model',
        title=f'F1-Score Comparison Across GPUs (Batch Size = {stable_batch_size})'
    )
    fig.show()
else:
    print("F1 score data not found for accuracy comparison across hardware.")

## 7. Performance vs. Accuracy Trade-off

Finally, we create a scatter plot to visualize the trade-off between throughput and F1-score. Each point represents a different experiment configuration, helping us identify the sweet spot for performance and accuracy.

In [ ]:
if 'f1_score' in df.columns and 'throughput' in df.columns:
    tradeoff_df = df[df['f1_score'].notna() & df['throughput'].notna()]
    fig = px.scatter(
        tradeoff_df,
        x='f1_score',
        y='throughput',
        color='precision',
        symbol='gpu_type',
        size='batch_size',
        facet_col='model',
        title='Performance vs. Accuracy Trade-off',
        hover_data=['batch_size', 'gpu_type', 'precision']
    )
    fig.update_layout(height=600)
    fig.show()
else:
    print("Not enough data for a trade-off plot (missing F1 score or throughput).")

NameError: name 'df' is not defined